# Пробуем переделать под использование Dataloader

In [75]:
import itertools
import random
import string
from collections import Counter
from itertools import chain

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import umap
from IPython.display import clear_output
from matplotlib import pyplot as plt
from nltk.tokenize import WordPunctTokenizer
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
from tqdm.auto import tqdm as tqdma

## Загрузка данных

In [76]:
data = list(open("./quora.txt", encoding="utf-8"))
data[50]

"What TV shows or books help you read people's body language?\n"

In [77]:
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

['What', 'TV', 'shows', 'or', 'books', 'help', 'you', 'read', 'people', "'", 's', 'body', 'language', '?']


In [78]:
data_tok = [
    tokenizer.tokenize(
        line.translate(str.maketrans("", "", string.punctuation)).lower()
    )
    for line in data
]
data_tok = [x for x in data_tok if len(x) >= 3]

In [79]:
min_count = 5
window_radius = 5

In [80]:
vocabulary_with_counter = Counter(chain.from_iterable(data_tok))

word_count_dict = dict()
for word, counter in vocabulary_with_counter.items():
    if counter >= min_count:
        word_count_dict[word] = counter

vocabulary = set(word_count_dict.keys())
del vocabulary_with_counter

In [81]:
word_to_index = {word: index for index, word in enumerate(vocabulary)}
index_to_word = {index: word for word, index in word_to_index.items()}

In [82]:
context_pairs = []

for text in data_tok:
    for i, central_word in enumerate(text):
        context_indices = range(
            max(0, i - window_radius), min(i + window_radius, len(text))
        )
        for j in context_indices:
            if j == i:
                continue
            context_word = text[j]
            if central_word in vocabulary and context_word in vocabulary:
                context_pairs.append(
                    (word_to_index[central_word], word_to_index[context_word])
                )

print(f"Generated {len(context_pairs)} pairs of target and context words.")

Generated 40220313 pairs of target and context words.


In [83]:
def subsample_frequent_words(word_count_dict, threshold = 1e-5):  # threshold=8e-3

    total_words = sum(word_count_dict.values())
    keep_prob_dict = {}

    for word, count in word_count_dict.items():
        frequency = count / total_words
        # word_prob = ((count / (threshold * total_words))**0.5 + 1) * (threshold * total_words) / count
        # word_prob = 1 - (threshold / frequency)**0.5
        word_prob = (threshold / frequency)**0.5
        keep_prob_dict[word] = word_prob

    return keep_prob_dict

In [84]:
def get_negative_sampling_prob(word_count_dict):
    
    total_count = sum(word_count_dict.values())
    unigram_probs = {word: count / total_count for word, count in word_count_dict.items()}
    
    # Вычисляем нормировочную константу Z
    Z = sum([prob ** 0.75 for prob in unigram_probs.values()])
    
    negative_sampling_prob_dict = {word: (prob ** 0.75) / Z for word, prob in unigram_probs.items()} 

    return negative_sampling_prob_dict

Для удобства, преобразуем полученные словари в массивы (т.к. все слова все равно уже пронумерованы).

In [85]:
keep_prob_dict = subsample_frequent_words(word_count_dict)
assert keep_prob_dict.keys() == word_count_dict.keys()

In [86]:
negative_sampling_prob_dict = get_negative_sampling_prob(word_count_dict)
assert negative_sampling_prob_dict.keys() == negative_sampling_prob_dict.keys()
assert np.allclose(sum(negative_sampling_prob_dict.values()), 1)

In [87]:
keep_prob_array = np.array(
    [keep_prob_dict[index_to_word[idx]] for idx in range(len(word_to_index))]
)
negative_sampling_prob_array = np.array(
    [
        negative_sampling_prob_dict[index_to_word[idx]]
        for idx in range(len(word_to_index))
    ]
)

Если все прошло успешно, функция ниже поможет вам с генерацией подвыборок (батчей).

In [88]:
def generate_batch_with_neg_samples(
    context_pairs,
    batch_size,
    keep_prob_array,
    word_to_index,
    num_negatives,
    negative_sampling_prob_array,
):
    batch = []
    neg_samples = []

    while len(batch) < batch_size:
        center, context = random.choice(context_pairs)
        if random.random() < keep_prob_array[center]:
            batch.append((center, context))
            neg_sample = np.random.choice(
                range(len(negative_sampling_prob_array)),
                size=num_negatives,
                p=negative_sampling_prob_array,
            )
            neg_samples.append(neg_sample)
    batch = np.array(batch)
    neg_samples = np.vstack(neg_samples)
    return batch, neg_samples

In [89]:
batch_size = 4
num_negatives = 15
batch, neg_samples = generate_batch_with_neg_samples(
    context_pairs,
    batch_size,
    keep_prob_array,
    word_to_index,
    num_negatives,
    negative_sampling_prob_array,
)

Определим датасет и загрузчик данных:

In [90]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, context_pairs, keep_prob_array, num_negatives, negative_sampling_prob_array):
        self.context_pairs = context_pairs
        self.keep_prob_array = keep_prob_array
        self.num_negatives = num_negatives
        self.negative_sampling_prob_array = negative_sampling_prob_array

    def __len__(self):
        return len(self.context_pairs)

    def __getitem__(self, idx):
        center, context = self.context_pairs[idx]
        if np.random.rand() < self.keep_prob_array[center]:
            neg_samples = np.random.choice(
                range(len(self.negative_sampling_prob_array)),
                size=self.num_negatives,
                p=self.negative_sampling_prob_array,
            )
            return torch.tensor(center), torch.tensor(context), torch.tensor(neg_samples)
        else:
            return None

def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    if len(batch) == 0:
        return None
    centers, contexts, neg_samples = zip(*batch)
    return torch.stack(centers), torch.stack(contexts), torch.stack(neg_samples)

Определим модель:

In [91]:
class SkipGramModelWithNegSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embeddings = torch.nn.Embedding(vocab_size, embedding_dim) # Попробовать sparse=True
        self.context_embeddings = torch.nn.Embedding(vocab_size, embedding_dim) # Попробовать sparse=True

    def forward(self, center_words, pos_context_words, neg_context_words):
        center_embeds = self.center_embeddings(center_words)  # batch size, embedding_dim
        pos_context_embeds = self.context_embeddings(pos_context_words)  # batch size, embedding dim

        neg_context_embeds = self.context_embeddings(neg_context_words)  # batch_size, num_negatives, embedding_dim

        pos_dot_products = torch.sum(center_embeds * pos_context_embeds, dim=-1)  # batch_size
        pos_scores = F.logsigmoid(pos_dot_products)  # Применяем логсигмоиду к pos_scores

        center_embeds_expanded = center_embeds.unsqueeze(1)  # batch_size, 1, embedding_dim

        neg_dot_products = torch.bmm(neg_context_embeds, center_embeds_expanded.transpose(1, 2)).squeeze(2)  # batch_size, num_negatives
        neg_scores = F.logsigmoid(neg_dot_products)  # Применяем логсигмоиду к neg_scores УБРАЛ - !!!!

        return pos_scores, neg_scores

In [92]:
vocab_size = len(word_to_index)
embedding_dim = 32
num_negatives = 15

In [93]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SkipGramModelWithNegSampling(vocab_size, embedding_dim).to(device)

Функция обучения с загрузчиком данных:

In [94]:
optimizer = optim.Adam(model.parameters(), lr=0.05) # Попробовать SparceAdam с установкой в слоях эмбеддингов sparse=True
lr_scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=150)
criterion = nn.BCEWithLogitsLoss()

In [95]:
from torch.utils.data import DataLoader

def train_skipgram_with_neg_sampling(
    model,
    context_pairs,
    keep_prob_array,
    batch_size,
    num_negatives,
    negative_sampling_prob_array,
    steps,
    optimizer,
    lr_scheduler,
    device,
):
    dataset = CustomDataset(
        context_pairs, keep_prob_array, num_negatives, negative_sampling_prob_array
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)

    # neg_labels теперь создается с размером (batch_size * num_negatives)
    neg_labels = torch.zeros(batch_size * num_negatives).to(device) 
    loss_history = []

    for step in tqdma(range(steps)):
        for batch in dataloader:
            if batch is None:
                continue

            centers, contexts, neg_samples = batch
            centers = centers.to(device)
            contexts = contexts.to(device)
            neg_samples = neg_samples.to(device)

            optimizer.zero_grad()
            pos_scores, neg_scores = model(centers, contexts, neg_samples)

            # Изменяем размер neg_scores для соответствия neg_labels
            neg_scores = neg_scores.view(-1)  

            # Создаем pos_labels с динамическим размером
            pos_labels = torch.ones(pos_scores.size(0)).to(device)  

            loss_pos = criterion(pos_scores, pos_labels)
            loss_neg = criterion(neg_scores, neg_labels)

            loss = loss_pos + loss_neg
            loss.backward()
            optimizer.step()

            loss_history.append(loss.item())
            lr_scheduler.step(loss_history[-1])

            if step % 100 == 0:
                print(
                    f"Step {step}, Loss: {np.mean(loss_history[-100:])}, learning rate: {lr_scheduler._last_lr}"
                )

Запуск обучения:

In [104]:
steps = 500    # steps = 2500
batch_size = 1 # batch_size = 512

optimizer = optim.Adam(model.parameters(), lr=0.05) # Попробовать SparceAdam с установкой в слоях эмбеддингов sparse=True
lr_scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=150)
criterion = nn.BCEWithLogitsLoss()

train_skipgram_with_neg_sampling(
    model=model,
    context_pairs=context_pairs,
    keep_prob_array=keep_prob_array,
    batch_size=batch_size,
    num_negatives=num_negatives,
    negative_sampling_prob_array=negative_sampling_prob_array,
    steps=steps,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    device=device  # 'cuda' если доступна GPU, иначе 'cpu'
)

  0%|          | 0/500 [00:00<?, ?it/s]

Step 0, Loss: 4.508932113647461, learning rate: [0.05]
Step 0, Loss: 5.6592864990234375, learning rate: [0.05]
Step 0, Loss: 4.12333345413208, learning rate: [0.05]
Step 0, Loss: 3.4721107482910156, learning rate: [0.05]
Step 0, Loss: 3.097061204910278, learning rate: [0.05]
Step 0, Loss: 2.7544604738553367, learning rate: [0.05]
Step 0, Loss: 2.5374675818852017, learning rate: [0.05]
Step 0, Loss: 3.4723781645298004, learning rate: [0.05]
Step 0, Loss: 3.6200896633995905, learning rate: [0.05]
Step 0, Loss: 3.7803043127059937, learning rate: [0.05]
Step 0, Loss: 4.1398068558086045, learning rate: [0.05]
Step 0, Loss: 4.326871931552887, learning rate: [0.05]
Step 0, Loss: 4.621867014811589, learning rate: [0.05]
Step 0, Loss: 4.366476195199149, learning rate: [0.05]
Step 0, Loss: 4.826655133565267, learning rate: [0.05]
Step 0, Loss: 4.590424470603466, learning rate: [0.05]
Step 0, Loss: 4.539224281030543, learning rate: [0.05]
Step 0, Loss: 4.637637899981605, learning rate: [0.05]
Ste

KeyboardInterrupt: 